# 🎓 SI-REKTA: Sistem Informasi Rekomendasi Dosen Pembimbing Tugas Akhir Berbasis Machine Learning & NLP

> **Mata Kuliah:** Machine Learning (Semester 5) — Departemen Sistem Informasi  
> **Dosen Pengampu:** Aina Hubby Aziira, M.Eng  
> **Metodologi Pengembangan:** CRISP-ML(Q) (*Cross-Industry Standard Process for Data Mining*)  
> **Repository GitHub:** [https://github.com/accfd/SI-REKTA](https://github.com/accfd/SI-REKTA)  
> **Dokumen Rujukan:**  
> - 📄 Tinjauan Pustaka Sistematis: [`docs/SI-REKTA_SLR.md`](docs/SI-REKTA_SLR.md)  
> - 🗺️ Blueprint Metodologi CRISP-ML(Q): [`docs/SI-REKTA_CRISP_ML_Q.md`](docs/SI-REKTA_CRISP_ML_Q.md)  

---

## 📌 Deskripsi & Tujuan Proyek

Notebook ini merupakan implementasi teknis dari proyek **SI-REKTA** (*Sistem Informasi Rekomendasi Dosen Pembimbing Tugas Akhir*). Masalah utama yang diselesaikan adalah **ketidakcocokan (*mismatch*) antara rencana topik tugas akhir mahasiswa dengan kepakaran riset dosen**, serta **proses penentuan (*plotting*) pembimbing yang memakan waktu lama dan kerap memicu ketimpangan beban kerja dosen (*workload imbalance*)**.

Melalui pendekatan *Natural Language Processing* (NLP) dan *Supervised Machine Learning*, notebook ini memproses data historis 625 judul skripsi dengan karakteristik teks pendek (rata-rata 16 kata) dan ketimpangan kelas yang tajam (satu dosen memiliki 145 bimbingan vs 10–30 bimbingan dosen lain). Solusi komputasional yang diterapkan mengombinasikan ekstraksi fitur **TF-IDF Word (1–2 gram) + Character (3–5 gram)**, klasifikasi **Linear Support Vector Machine (Linear SVM)** berbobot seimbang (*balanced class weighting*), serta **Hybrid Cosine Similarity Gate** terhadap profil rekam jejak riset dosen untuk menghasilkan rekomendasi **Top-3 Dosen Pembimbing** yang paling relevan, adil, dan terukur.

Seluruh eksperimen di dalam notebook ini disusun secara sistematis mengacu pada 6 fase **CRISP-ML(Q)** dengan pendekatan **Self-Explaining Notebook**: setiap sel kode selalu diawali dengan landasan teoritis (*"Mengapa tahap ini dilakukan?"*) dan diakhiri dengan evaluasi hasil (*"Interpretasi Output"*).

---

## 📑 Daftar Isi Alur Kerja CRISP-ML(Q):
1. **[Fase 1: Business Understanding](#fase-1-business-understanding)** — Identifikasi masalah, tujuan proyek, dan metrik keberhasilan.
2. **[Fase 2: Data Understanding](#fase-2-data-understanding)** — Pemuatan dataset, Exploratory Data Analysis (EDA), dan audit kualitas data riil.
3. **[Fase 3: Data Preparation](#fase-3-data-preparation)** — Entity canonicalization nama dosen, pembersihan teks, stopword akademik, stemming, dan ekstraksi TF-IDF n-gram.
4. **[Fase 4: Modeling](#fase-4-modeling)** — Pelatihan Baseline Naive Bayes, Main Model Linear SVM (Balanced), dan pembentukan matriks profil Cosine Similarity.
5. **[Fase 5: Evaluation](#fase-5-evaluation)** — Validasi silang Stratified 5-Fold CV, Top-1 & Top-3 Accuracy (Hit@3), Macro-F1 Score, serta analisis keadilan kelas.
6. **[Fase 6: Deployment Preparation](#fase-6-deployment-preparation)** — Serialisasi pipeline mandiri ke `si_rekta_pipeline.joblib` dan simulasi inferensi judul baru.

\n

# <a id="fase-1-business-understanding"></a>1. Fase 1: Business Understanding

> **Referensi Ilmiah:** [`docs/SI-REKTA_SLR.md`](docs/SI-REKTA_SLR.md) Bab 1 & Bab 3

---

### 1.1 Latar Belakang & Identifikasi Masalah Organisasi
Pada Departemen Sistem Informasi, pemilihan dan penentuan dosen pembimbing tugas akhir merupakan proses krusial yang menentukan kelancaran studi mahasiswa. Berdasarkan analisis permasalahan di lapangan:
1. **Ketidaksesuaian Topik (*Topic Mismatch*):** Mahasiswa kerap memilih dosen pembimbing berdasarkan faktor subjektif (popularitas atau kemudahan dihubungi) tanpa mempertimbangkan keselarasan dengan peta jalan (*roadmap*) riset dosen. Hal ini berisiko menyebabkan usulan proposal ditolak oleh komisi tugas akhir atau memerlukan revisi substansial.
2. **Beban Bimbingan Timpang (*Workload Imbalance*):** Penentuan manual oleh koordinator tugas akhir menimbulkan konsentrasi bimbingan ekstrem pada segelintir dosen (misal: 145 bimbingan pada satu dosen vs 10–30 bimbingan pada dosen lain).
3. **Kebutuhan Sistem Pendukung Keputusan:** Diperlukan sistem rekomendasi cerdas yang mampu menganalisis kesesuaian semantik draf judul tugas akhir secara otomatis, objektif, dan terintegrasi ke dalam aplikasi web.

---

### 1.2 Tujuan Proyek (Business & Technical Goals)
* **Tujuan Bisnis:** Membantu mahasiswa menemukan calon pembimbing yang paling relevan secara mandiri serta mempermudah koordinator tugas akhir dalam memverifikasi plotting pembimbing secara adil.
* **Tujuan Teknis ML:** Membangun pipeline NLP dan model klasifikasi multi-kelas berbasis teks judul tugas akhir (bahasa Indonesia) yang mampu memprediksi probabilitas kecocokan dosen dan menyajikan peringkat **Top-3 Dosen Pembimbing** yang paling sesuai.

---

### 1.3 Kriteria Keberhasilan & Indikator Kinerja (KPI)
Berdasarkan sintesis 13 artikel ilmiah bereputasi pada [`docs/SI-REKTA_SLR.md`](docs/SI-REKTA_SLR.md), kriteria keberhasilan model ditetapkan sebagai berikut:
1. **Top-3 Accuracy (Hit Rate @ 3) $\ge$ 80% (0,80):** Dosen pembimbing yang tepat wajib berada dalam 3 rekomendasi teratas sistem minimal pada 80% data uji.
2. **Macro-Averaged F1-Score $\ge$ 70% (0,70):** Menjamin model memiliki presisi dan recall yang adil di seluruh kelas dosen (termasuk dosen kelas minoritas).
3. **Mean Reciprocal Rank (MRR @ 3) $\ge$ 0,60:** Mengukur ketepatan urutan peringkat rekomendasi (posisi 1, 2, atau 3).
4. **Deployability:** Model dapat diekspor (`.joblib`) dan memiliki waktu respons inferensi di bawah 1 detik (*sub-second latency*) untuk integrasi API web.
\n

---

## 🛠️ Pemuatan Pustaka (Library Importation) & Konfigurasi Lingkungan

### 💡 Mengapa Tahap Ini Dilakukan?
Sebelum melakukan manipulasi data dan pemodelan, kita perlu memuat seluruh pustaka (*dependencies*) pendukung yang telah dirancang pada dokumen metodologi [`docs/SI-REKTA_CRISP_DM.md`](docs/SI-REKTA_CRISP_DM.md):
* **Manipulasi & Analisis Data:** `pandas` dan `numpy` untuk operasi tabel data dan manipulasi array numerik.
* **Pembersihan Teks & Regex:** `re`, `string`, dan `collections` untuk prapengolahan string bahasa Indonesia dan analisis frekuensi istilah.
* **Ekstraksi Fitur Teks:** `TfidfVectorizer` dari Scikit-Learn untuk mentransformasi teks judul menjadi representasi numerik berbasis n-gram leksikal dan karakter.
* **Algoritma Machine Learning:** 
  * `MultinomialNB` sebagai model pembanding (*baseline wajib* mengacu pada Sinaga & Kurniawan, 2025).
  * `LinearSVC` dan `CalibratedClassifierCV` sebagai model utama (*main model* berbobot seimbang mengacu pada Bolshakov, 2025).
* **Evaluasi & Pengujian:** `StratifiedKFold`, `train_test_split`, `classification_report`, `confusion_matrix`, dan fungsi metrik ranking.
* **Pengukuran Kemiripan Semantik:** `cosine_similarity` untuk modul *hybrid similarity gate*.
* **Visualisasi Data:** `matplotlib.pyplot` dan `seaborn` untuk pembuatan grafik EDA dan matriks konfusi.
* **Penyimpanan Pipeline:** `joblib` untuk serialisasi model akhir.
\n

In [ ]:
# Pemuatan pustaka standar sistem
import os
import re
import string
import warnings
from collections import Counter

# Pustaka pengolahan data & komputasi numerik
import numpy as np
import pandas as pd

# Pustaka visualisasi data
import matplotlib.pyplot as plt
import seaborn as sns

# Pustaka Scikit-Learn: Text Preprocessing & Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Pustaka Scikit-Learn: Modeling & Calibration
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline

# Pustaka Scikit-Learn: Evaluation & Validation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Pustaka serialisasi model
import joblib

# Konfigurasi visualisasi & warning
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print("✅ Seluruh pustaka berhasil dimuat dengan sukses!")
\n

### 🔍 Interpretasi Hasil Setup Lingkungan
Seluruh pustaka pendukung (*stack*) ilmu data, pemrosesan teks, pemodelan linier berbobot seimbang, dan evaluasi telah berhasil dimuat tanpa kendala dependensi. Lingkungan komputasi kini siap untuk melanjutkan ke **Fase 2: Data Understanding** (pemuatan dataset `docs/Judul_TA.csv` dan analisis data eksploratif).
\n